In [ ]:
#python -m venv openai-env
#openai-env\Scripts\activate
#API设置：https://platform.openai.com/docs/quickstart
#pip install ipykernel
#python -m ipykernel install --user --name=openai-env --display-name "Python (openai-env)"
#jupyter notebook
#需要绑定付款方式：https://platform.openai.com/settings/organization/billing/overview

In [1]:
#!pip install --upgrade openai

In [2]:
!pip show openai

Name: openai
Version: 0.27.4
Summary: Python client library for the OpenAI API
Home-page: https://github.com/openai/openai-python
Author: OpenAI
Author-email: support@openai.com
License: 
Location: c:\users\yurul\anaconda3\lib\site-packages
Requires: aiohttp, requests, tqdm
Required-by: 


In [1]:
import base64
import requests
from openai import OpenAI
import os
import re

In [ ]:
api_key=os.environ.get("OPENAI_API_KEY")
print(api_key)

In [ ]:
# Function to encode the image
def encode_image(image_path):
  with open(image_path, "rb") as image_file:
    return base64.b64encode(image_file.read()).decode('utf-8')

# analyze images in folder

In [ ]:
import os
import csv
import requests
import base64
import re
import time

# Function to encode image to base64
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def analyze_image(image_path, api_key, timeout=10):
    base64_image = encode_image(image_path)
    # 设置请求头
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}"
    }

    # 设置请求负载
    payload = {
        "model": "gpt-4o",
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "does this photo include the logo of the green party in Germany, please only answer yes or no."
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}"
                        }
                    }
                ]
            }
        ],
        "max_tokens": 300
    }

    # 发送请求
    response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload)

    # 解析响应
    if response.status_code == 200:
        response_json = response.json()
        content = response_json['choices'][0]['message']['content']
        return content
    else:
        return f"Error: {response.status_code}, {response.text}"
    
# Function to read existing image names from the CSV file
def get_existing_images(output_csv):
    existing_images = set()  # Use a set for faster lookup
    if os.path.exists(output_csv):
        with open(output_csv, mode='r', encoding='utf-8') as file:
            reader = csv.reader(file)
            next(reader)  # Skip the header row
            for row in reader:
                if row:
                    existing_images.add(row[0])  # Add the image filename to the set
    return existing_images

# Function to process the image and retry in case of network failure
def analyze_image_with_retry(image_path, api_key, retries=3, delay=5, timeout=10):
    for attempt in range(retries):
        try:
            # 调用 analyze_image 函数处理图片，并设置超时时间
            category = analyze_image(image_path, api_key, timeout=timeout)
            if category is not None:
                return category
            else:
                print(f"Failed to get valid response for {os.path.basename(image_path)}.")
                return None, None
        except requests.exceptions.Timeout:
            # 如果是网络超时，等待一段时间并重试
            print(f"Request timed out for {os.path.basename(image_path)}. Retrying {attempt + 1}/{retries} after {delay} seconds...")
            time.sleep(delay)
        except requests.exceptions.RequestException as e:
            # 捕获其他网络错误
            print(f"Network error: {e}. Retrying {attempt + 1}/{retries} after {delay} seconds...")
            time.sleep(delay)
        except Exception as e:
            # 捕获其他错误
            print(f"Error processing {os.path.basename(image_path)}: {e}")
            return None, None
    return None, None  # 如果所有尝试都失败，返回None


# Function to process images in a folder and save results to CSV
def process_images(folder_path, api_key, output_csv):
    # Check if the file already exists to determine whether to write the header
    file_exists = os.path.exists(output_csv)

    # Get the set of existing images in the CSV file
    existing_images = get_existing_images(output_csv)

    # Open the CSV file in append mode
    with open(output_csv, mode='a', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)

        # If the file doesn't exist, write the header
        if not file_exists:
            writer.writerow(["Image", "if green party"])

        # Loop through the files in the folder
        for filename in os.listdir(folder_path):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                # Check if the image has already been processed
                if filename not in existing_images:
                    image_path = os.path.join(folder_path, filename)
                    try:
                        # Process the image and get the category and confidence (with retry)
                        category = analyze_image_with_retry(image_path, api_key)
                        if category:
                            writer.writerow([filename, category])
                            print(f"Processed {filename}: {category}")
                        else:
                            print(f"Failed to process {filename} after retries.")
                    except Exception as e:
                        print(f"Error processing {filename}: {e}")
                # 如果你仍然需要处理已存在的图片但不打印跳过信息，可以在这里添加代码
                # 否则，已跳过的图片将不会被处理

# Example usage:
# process_images("F:\\phd data\\post images", "your_openai_api_key", "F:\\test\\test.csv")


In [ ]:
# Main execution
if __name__ == "__main__":
    # Retrieve the API key from the environment variable
    api_key = os.getenv('OPENAI_API_KEY')

    # Ensure the API key is set
    if not api_key:
        raise ValueError("OpenAI API key is not set in the environment variable 'OPENAI_API_KEY'")

    # Define the folder path and output CSV file
    #folder_path = r"F:\phd data\post images"
    folder_path = r'F:\phd data\green party'
    output_csv = r"C:\coding\jupyternotebook\phd project\results\img_green party logo.csv"

    # Process the images and save the results to CSV
    process_images(folder_path, api_key, output_csv)